# CSQA Layer-Output Denoising And Activation Patching Analysis

Intervention notebook for clean-to-corrupted recovery analysis.

- model family: Qwen2.5
- target checkpoint: 3B instruct model
- corruption type: additive noise on one decoder layer output
- patch type: replacement of one decoder layer output with the clean activation
- decision space: constrained A-E answer-choice logits only

In [ ]:
import math

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

from src.data.load_csqa import load_csqa

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 220)


## Configuration

CORRUPTION_LAYER_1BASED = None resolves to the final decoder layer.


In [ ]:
MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"
EVAL_SPLIT = "validation"
EVAL_LIMIT = 64
MAX_SEQ_LEN = 384
CLEAN_BATCH_SIZE = 4
INTERVENTION_BATCH_SIZE = 2
CORRUPTION_NOISE_SCALE = 0.25
CORRUPTION_LAYER_1BASED = None
PATCH_LAYERS_1BASED = None
SEED = 42
USE_BFLOAT16_IF_AVAILABLE = True

## Environment Setup


In [ ]:
torch.manual_seed(SEED)
np.random.seed(SEED)

LETTERS = ["A", "B", "C", "D", "E"]

eval_rows = load_csqa(split=EVAL_SPLIT, limit=EVAL_LIMIT).copy()
eval_rows["n_choices"] = eval_rows["csqa_choices"].map(len)
eval_rows["prompt_len_chars"] = eval_rows["text"].str.len()
assert eval_rows["n_choices"].eq(5).all(), "Expected 5 choices for every CSQA row."

if torch.cuda.is_available():
    if USE_BFLOAT16_IF_AVAILABLE and torch.cuda.is_bf16_supported():
        model_dtype = torch.bfloat16
    else:
        model_dtype = torch.float16
    device_map = "auto"
else:
    model_dtype = torch.float32
    device_map = None

tok = AutoTokenizer.from_pretrained(MODEL_ID)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
tok.padding_side = "left"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=model_dtype,
    device_map=device_map,
    attn_implementation="eager",
)
model.eval()


def build_answer_token_ids(tok):
    out = {}
    for letter in LETTERS:
        ids = tok(" " + letter, add_special_tokens=False)["input_ids"]
        if len(ids) != 1:
            raise ValueError(f"Answer token '{letter}' is not single-token: {ids}")
        out[letter] = int(ids[0])
    return out


answer_token_ids = build_answer_token_ids(tok)
answer_ids = [answer_token_ids[l] for l in LETTERS]
answer_id_tensor = torch.tensor(answer_ids, dtype=torch.long)

display(eval_rows[["example_id", "answerKey", "prompt_len_chars"]].head())
print("eval rows:", len(eval_rows))
print("answer token ids:", answer_token_ids)


## Helper Functions


In [ ]:
def get_decoder_layers(model):
    candidates = [
        "model.layers",
        "transformer.h",
        "gpt_neox.layers",
    ]
    for path in candidates:
        cur = model
        ok = True
        for part in path.split("."):
            if not hasattr(cur, part):
                ok = False
                break
            cur = getattr(cur, part)
        if ok:
            return cur
    raise ValueError("Could not locate decoder layers on this model.")


def encode_prompts(texts, tok, max_seq_len):
    batch = tok(
        list(texts),
        add_special_tokens=False,
        truncation=True,
        max_length=max_seq_len,
        padding=True,
        return_tensors="pt",
    )
    pos = []
    for mask in batch["attention_mask"]:
        nz = torch.nonzero(mask, as_tuple=False).view(-1)
        pos.append(int(nz[-1].item()))
    batch["decision_pos"] = torch.tensor(pos, dtype=torch.long)
    return batch


def compute_choice_metrics(choice_logits, true_choice_idx):
    pred_idx = choice_logits.argmax(dim=-1)
    row_idx = torch.arange(choice_logits.shape[0], device=choice_logits.device)
    true_logits = choice_logits[row_idx, true_choice_idx]
    other_logits = choice_logits.clone()
    other_logits[row_idx, true_choice_idx] = -torch.inf
    best_other = other_logits.max(dim=-1).values
    correct_margin = true_logits - best_other
    return pred_idx.detach().cpu(), correct_margin.detach().cpu()


def apply_token_noise(hidden, decision_pos, noise_scale):
    row_idx = torch.arange(hidden.shape[0], device=hidden.device)
    token_hidden = hidden[row_idx, decision_pos]
    rms = token_hidden.float().pow(2).mean(dim=-1, keepdim=True).sqrt().to(token_hidden.dtype)
    noise = torch.randn_like(token_hidden) * (noise_scale * rms)
    hidden_out = hidden.clone()
    hidden_out[row_idx, decision_pos] = token_hidden + noise
    return hidden_out


def patch_token_hidden(hidden, decision_pos, clean_hidden):
    row_idx = torch.arange(hidden.shape[0], device=hidden.device)
    hidden_out = hidden.clone()
    hidden_out[row_idx, decision_pos] = clean_hidden.to(hidden.device, dtype=hidden.dtype)
    return hidden_out


decoder_layers = get_decoder_layers(model)
L = len(decoder_layers)
answer_ids_on_device = answer_id_tensor.to(model.lm_head.weight.device)

resolved_corruption_layer_1based = L if CORRUPTION_LAYER_1BASED is None else int(CORRUPTION_LAYER_1BASED)
if not (1 <= resolved_corruption_layer_1based <= L):
    raise ValueError("CORRUPTION_LAYER_1BASED is out of range.")

resolved_patch_layers_1based = list(range(1, L + 1)) if PATCH_LAYERS_1BASED is None else list(PATCH_LAYERS_1BASED)
resolved_patch_layers_1based = [int(x) for x in resolved_patch_layers_1based]
resolved_patch_layers_0based = [x - 1 for x in resolved_patch_layers_1based]
resolved_corruption_layer_0based = resolved_corruption_layer_1based - 1

print("decoder layers:", L)
print("resolved corruption layer:", resolved_corruption_layer_1based)
print("patch layers:", resolved_patch_layers_1based[:10], "..." if len(resolved_patch_layers_1based) > 10 else "")


## Data Generation And Extraction


### Clean Baseline And Clean Layer-Output Cache

Clean forward pass with extraction of the decision-position decoder layer outputs.


In [ ]:
def extract_clean_baseline_and_cache(frame):
    rows = []
    cache_blocks = []

    for start in tqdm(range(0, len(frame), CLEAN_BATCH_SIZE), total=int(math.ceil(len(frame) / CLEAN_BATCH_SIZE)), desc="clean baseline and cache"):
        batch_df = frame.iloc[start:start + CLEAN_BATCH_SIZE].reset_index(drop=True)
        batch_cpu = encode_prompts(batch_df["text"], tok, MAX_SEQ_LEN)
        decision_pos = batch_cpu.pop("decision_pos")
        batch = {k: v.to(model.device) for k, v in batch_cpu.items()}
        decision_pos = decision_pos.to(model.device)
        true_choice_idx = torch.tensor([LETTERS.index(str(x)) for x in batch_df["answerKey"].tolist()], dtype=torch.long, device=model.device)

        with torch.no_grad():
            out = model(**batch, output_hidden_states=True, return_dict=True, use_cache=False)

        row_idx = torch.arange(len(batch_df), device=decision_pos.device)
        final_logits = out.logits[row_idx, decision_pos][:, answer_ids_on_device].float()
        pred_idx, correct_margin = compute_choice_metrics(final_logits, true_choice_idx)
        clean_is_correct = pred_idx.eq(true_choice_idx.detach().cpu())

        per_layer_hidden = []
        for li in range(L):
            hidden = out.hidden_states[li + 1][row_idx, decision_pos].detach().cpu().to(torch.float16)
            per_layer_hidden.append(hidden)
        batch_cache = torch.stack(per_layer_hidden, dim=1)
        cache_blocks.append(batch_cache)

        for bi in range(len(batch_df)):
            rows.append(
                {
                    "example_id": batch_df.loc[bi, "example_id"],
                    "true_choice_idx": int(true_choice_idx[bi].item()),
                    "clean_prediction_idx": int(pred_idx[bi].item()),
                    "clean_prediction": LETTERS[int(pred_idx[bi].item())],
                    "clean_correct_margin": float(correct_margin[bi].item()),
                    "clean_is_correct": bool(clean_is_correct[bi].item()),
                }
            )

    return pd.DataFrame(rows), torch.cat(cache_blocks, dim=0)


clean_df, clean_layer_output_cache = extract_clean_baseline_and_cache(eval_rows)
analysis_df = eval_rows.merge(clean_df, on="example_id", how="left", validate="one_to_one")

display(
    analysis_df[["example_id", "answerKey", "clean_prediction", "clean_correct_margin", "clean_is_correct"]].head()
)
print("clean cache shape:", tuple(clean_layer_output_cache.shape))
print("clean accuracy:", round(float(analysis_df["clean_is_correct"].mean()), 4))


### Corrupted Baseline

Single corruption site with additive noise on the selected decoder layer output.


In [ ]:
def run_corrupted_layer_output_baseline(frame, target_layer_0based, noise_scale):
    rows = []

    def forward_seed(batch_start):
        torch.manual_seed(SEED + 1000 * (target_layer_0based + 1) + int(batch_start))

    current_state = {}

    def corruption_hook(module, args, output):
        hidden = output[0] if isinstance(output, tuple) else output
        hidden_out = apply_token_noise(hidden, current_state["decision_pos"], noise_scale)
        if isinstance(output, tuple):
            return (hidden_out,) + output[1:]
        return hidden_out

    handle = decoder_layers[target_layer_0based].register_forward_hook(corruption_hook)

    try:
        for start in tqdm(range(0, len(frame), INTERVENTION_BATCH_SIZE), total=int(math.ceil(len(frame) / INTERVENTION_BATCH_SIZE)), desc="corrupted baseline"):
            batch_df = frame.iloc[start:start + INTERVENTION_BATCH_SIZE].reset_index(drop=True)
            batch_cpu = encode_prompts(batch_df["text"], tok, MAX_SEQ_LEN)
            decision_pos = batch_cpu.pop("decision_pos")
            batch = {k: v.to(model.device) for k, v in batch_cpu.items()}
            decision_pos = decision_pos.to(model.device)
            true_choice_idx = torch.tensor([LETTERS.index(str(x)) for x in batch_df["answerKey"].tolist()], dtype=torch.long, device=model.device)

            current_state["decision_pos"] = decision_pos
            forward_seed(start)

            with torch.no_grad():
                out = model(**batch, return_dict=True, use_cache=False)

            row_idx = torch.arange(len(batch_df), device=decision_pos.device)
            final_logits = out.logits[row_idx, decision_pos][:, answer_ids_on_device].float()
            pred_idx, correct_margin = compute_choice_metrics(final_logits, true_choice_idx)
            corrupted_is_correct = pred_idx.eq(true_choice_idx.detach().cpu())

            for bi in range(len(batch_df)):
                rows.append(
                    {
                        "example_id": batch_df.loc[bi, "example_id"],
                        "corrupted_prediction_idx": int(pred_idx[bi].item()),
                        "corrupted_prediction": LETTERS[int(pred_idx[bi].item())],
                        "corrupted_correct_margin": float(correct_margin[bi].item()),
                        "corrupted_is_correct": bool(corrupted_is_correct[bi].item()),
                    }
                )
    finally:
        handle.remove()

    return pd.DataFrame(rows)


corrupted_df = run_corrupted_layer_output_baseline(
    eval_rows,
    resolved_corruption_layer_0based,
    CORRUPTION_NOISE_SCALE,
)

analysis_df = analysis_df.merge(corrupted_df, on="example_id", how="left", validate="one_to_one")
analysis_df["corrupted_prediction_preserved"] = analysis_df["corrupted_prediction_idx"].eq(analysis_df["clean_prediction_idx"])
analysis_df["corrupted_margin_delta"] = analysis_df["corrupted_correct_margin"] - analysis_df["clean_correct_margin"]
analysis_df["clean_correct_broken"] = analysis_df["clean_is_correct"] & (~analysis_df["corrupted_is_correct"])

corrupted_baseline_summary = pd.DataFrame(
    {
        "corruption_layer_1based": [resolved_corruption_layer_1based],
        "mean_corrupted_margin_delta": [float(analysis_df["corrupted_margin_delta"].mean())],
        "corrupted_prediction_preservation_rate": [float(analysis_df["corrupted_prediction_preserved"].mean())],
        "corrupted_accuracy": [float(analysis_df["corrupted_is_correct"].mean())],
        "clean_correct_break_rate": [float(analysis_df.loc[analysis_df["clean_is_correct"], "clean_correct_broken"].mean())],
    }
)

display(corrupted_baseline_summary.round(4))


### Layer-Output Activation Patching

Clean decoder layer outputs are patched into the corrupted run one patch layer at a time.


In [ ]:
def run_layer_output_activation_patching_scan(frame, clean_cache, target_corruption_layer_0based, patch_layers_0based, noise_scale):
    rows = []

    def forward_seed(batch_start):
        torch.manual_seed(SEED + 1000 * (target_corruption_layer_0based + 1) + int(batch_start))

    for patch_layer_0based in tqdm(patch_layers_0based, total=len(patch_layers_0based), desc="layer-output activation patching"):
        current_state = {}

        def corruption_hook(module, args, output):
            hidden = output[0] if isinstance(output, tuple) else output
            hidden_out = apply_token_noise(hidden, current_state["decision_pos"], noise_scale)
            if isinstance(output, tuple):
                return (hidden_out,) + output[1:]
            return hidden_out

        def patch_hook(module, args, output):
            hidden = output[0] if isinstance(output, tuple) else output
            hidden_out = patch_token_hidden(hidden, current_state["decision_pos"], current_state["clean_patch_hidden"])
            if isinstance(output, tuple):
                return (hidden_out,) + output[1:]
            return hidden_out

        corruption_handle = decoder_layers[target_corruption_layer_0based].register_forward_hook(corruption_hook)
        patch_handle = decoder_layers[patch_layer_0based].register_forward_hook(patch_hook)

        try:
            for start in range(0, len(frame), INTERVENTION_BATCH_SIZE):
                batch_df = frame.iloc[start:start + INTERVENTION_BATCH_SIZE].reset_index(drop=True)
                batch_cpu = encode_prompts(batch_df["text"], tok, MAX_SEQ_LEN)
                decision_pos = batch_cpu.pop("decision_pos")
                batch = {k: v.to(model.device) for k, v in batch_cpu.items()}
                decision_pos = decision_pos.to(model.device)
                true_choice_idx = torch.tensor([LETTERS.index(str(x)) for x in batch_df["answerKey"].tolist()], dtype=torch.long, device=model.device)

                current_state["decision_pos"] = decision_pos
                current_state["clean_patch_hidden"] = clean_cache[start:start + len(batch_df), patch_layer_0based, :]
                forward_seed(start)

                with torch.no_grad():
                    out = model(**batch, return_dict=True, use_cache=False)

                row_idx = torch.arange(len(batch_df), device=decision_pos.device)
                final_logits = out.logits[row_idx, decision_pos][:, answer_ids_on_device].float()
                pred_idx, correct_margin = compute_choice_metrics(final_logits, true_choice_idx)
                patched_is_correct = pred_idx.eq(true_choice_idx.detach().cpu())

                for bi in range(len(batch_df)):
                    rows.append(
                        {
                            "example_id": batch_df.loc[bi, "example_id"],
                            "patch_layer": int(patch_layer_0based),
                            "patched_prediction_idx": int(pred_idx[bi].item()),
                            "patched_prediction": LETTERS[int(pred_idx[bi].item())],
                            "patched_correct_margin": float(correct_margin[bi].item()),
                            "patched_is_correct": bool(patched_is_correct[bi].item()),
                        }
                    )
        finally:
            patch_handle.remove()
            corruption_handle.remove()

    return pd.DataFrame(rows)


patching_df = run_layer_output_activation_patching_scan(
    eval_rows,
    clean_layer_output_cache,
    resolved_corruption_layer_0based,
    resolved_patch_layers_0based,
    CORRUPTION_NOISE_SCALE,
)

patching_df = patching_df.merge(
    analysis_df[[
        "example_id",
        "clean_prediction_idx",
        "clean_correct_margin",
        "clean_is_correct",
        "corrupted_prediction_idx",
        "corrupted_correct_margin",
        "corrupted_is_correct",
        "clean_correct_broken",
    ]],
    on="example_id",
    how="left",
    validate="many_to_one",
)

patching_df["patch_layer_1based"] = patching_df["patch_layer"] + 1
patching_df["margin_recovery"] = patching_df["patched_correct_margin"] - patching_df["corrupted_correct_margin"]
patching_df["patch_prediction_matches_clean"] = patching_df["patched_prediction_idx"].eq(patching_df["clean_prediction_idx"])
patching_df["broken_case_rescued"] = patching_df["clean_correct_broken"] & patching_df["patched_is_correct"]

display(patching_df.head())


## Basic Analysis


### Corrupted Baseline Summary


In [ ]:
display(corrupted_baseline_summary.round(4))


### Layer-Output Patching Summary


In [ ]:
patching_summary = (
    patching_df.groupby("patch_layer_1based")
    .agg(
        mean_patched_correct_margin=("patched_correct_margin", "mean"),
        mean_margin_recovery=("margin_recovery", "mean"),
        patch_prediction_matches_clean_rate=("patch_prediction_matches_clean", "mean"),
        patched_accuracy=("patched_is_correct", "mean"),
        broken_case_rescue_rate=("broken_case_rescued", "mean"),
    )
    .reset_index()
)

display(patching_summary.round(4))


In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(12, 16), sharex=True)

axes[0].plot(patching_summary["patch_layer_1based"], patching_summary["mean_margin_recovery"], marker="o")
axes[0].axhline(0.0, color="black", linewidth=1, alpha=0.6)
axes[0].set_ylabel("Mean recovery")
axes[0].set_title("Mean recovery of correct-answer margin by patch layer")

axes[1].plot(patching_summary["patch_layer_1based"], patching_summary["patch_prediction_matches_clean_rate"], marker="o")
axes[1].set_ylabel("Dataset fraction")
axes[1].set_title("Clean-prediction match rate after activation patching")

axes[2].plot(patching_summary["patch_layer_1based"], patching_summary["patched_accuracy"], marker="o")
axes[2].set_ylabel("Dataset fraction")
axes[2].set_title("Patched-run accuracy by patch layer")

axes[3].plot(patching_summary["patch_layer_1based"], patching_summary["broken_case_rescue_rate"], marker="o")
axes[3].set_xlabel("Patch layer")
axes[3].set_ylabel("Dataset fraction")
axes[3].set_title("Rescue rate on clean-correct corrupted cases by patch layer")

plt.tight_layout()
plt.show()


In [ ]:
display(
    patching_summary.sort_values(
        ["broken_case_rescue_rate", "mean_margin_recovery"],
        ascending=[False, False],
    )
    .head(10)
    .round(4)
)
